In [ ]:
import os
from rdkit import Chem
from rdkit.Chem import Draw
from rdkit.Chem import AllChem

# 1. Correct the path (Double check the spelling of your folders!)
# Fix the typo in the name if it was meant to be 13027549 instead of 130275249
# Use the direct absolute Windows path with an 'r' prefix to handle backslashes safely
file_path = r"D:/PriceTracker/ERBB2/data/prepared_ligands_pdb/ligand_130275249.pdb"

if not os.path.exists(file_path):
    raise FileNotFoundError(f"Could not find the file at: {os.path.abspath(file_path)}. Please verify the folder names.")

# 2. Load the PDB file sanitizing the structure safely
# Proximity bonding helps RDKit guess bond orders from distances since PDBs lack them
ligand = Chem.MolFromPDBFile(file_path, removeHs=False, sanitize=True, proximityBonding=True)

if ligand is None:
    raise ValueError("RDKit failed to parse the PDB file. Try converting the ligand to .sdf format if possible.")

# 3. Compute clean 2D coordinates for the publication map layout
AllChem.Compute2DCoords(ligand)

# 4. Generate and save the clean 2D image
drawer = Draw.MolDraw2DCairo(600, 600)
options = drawer.drawOptions()
options.backgroundColour = (1, 1, 1, 1)  # Clean white background
options.addAtomIndices = False

drawer.DrawMolecule(ligand)
drawer.FinishDrawing()

output_image = "ligand_2d_structure.png"
with open(output_image, "wb") as f:
    f.write(drawer.GetText())

print(f"Success! Saved 2D diagram to {output_image}")

OSError: Bad input file D:/PriceTracker/ERBB2/data/prepared_ligands_pdb/ligand_130275249.pdb

In [14]:
import os
from rdkit import Chem
from rdkit.Chem import Draw
from rdkit.Chem import AllChem

# Direct absolute path to your file
file_path = r"D:\PriceTracker\ERBB2\data\prepaerd_ligands_pdb\ligand_130275249_temp.sd"

print(f"Loading ligand from: {file_path}")

# Load the molecule from the SDF structure
supplier = Chem.SDMolSupplier(file_path)
ligand = supplier[0] if supplier else None

if ligand is None:
    raise ValueError("RDKit could not read the structure from the .sd file.")

# Compute clean 2D coordinates for the flat layout
AllChem.Compute2DCoords(ligand)

# Setup the 2D canvas drawing options
drawer = Draw.MolDraw2DCairo(600, 600)  # Width, Height
options = drawer.drawOptions()
options.backgroundColour = (1, 1, 1, 1)  # White background
options.addAtomIndices = False          # Clean structure layout

# Render the molecule structure
drawer.DrawMolecule(ligand)
drawer.FinishDrawing()

# FIXED LINE: Using GetDrawingText() for Cairo drawer objects
output_image = "ligand_130275249_2d.png"
with open(output_image, "wb") as f:
    f.write(drawer.GetDrawingText())

print(f"🎉 Success! Your 2D diagram has been saved as: {os.path.abspath(output_image)}")

Loading ligand from: D:\PriceTracker\ERBB2\data\prepaerd_ligands_pdb\ligand_130275249_temp.sd
🎉 Success! Your 2D diagram has been saved as: d:\PriceTracker\ERBB2\Codes\ligand_130275249_2d.png


In [23]:
import os
import MDAnalysis as mda
import prolif as plf
from rdkit import Chem

# 1. Define paths to your receptor and your docked ligand
receptor_path = r"D:\PriceTracker\ERBB2\data\receptor\3RCD.pdb" 
ligand_path = r"D:\PriceTracker\ERBB2\data\prepaerd_ligands_pdb\ligand_130275249_temp.sd"

if not os.path.exists(receptor_path) or not os.path.exists(ligand_path):
    raise FileNotFoundError("Please ensure both your receptor.pdb and ligand file paths are correct.")

print("Parsing structures using ProLIF native readers...")

# 2. Load the receptor into a ProLIF Molecule object
# FIXED: Added force=True to bypass the explicit hydrogen structural check
u_receptor = mda.Universe(receptor_path)
receptor_mol = plf.Molecule.from_mda(u_receptor, force=True)

# 3. Load the ligand from the .sd file using RDKit, then wrap with ProLIF
supplier = Chem.SDMolSupplier(ligand_path)
rdkit_ligand = supplier[0] if supplier else None
if rdkit_ligand is None:
    raise ValueError("RDKit failed to read the .sd file. Ensure it is a valid SDF format.")

# Convert RDKit object safely into a ProLIF molecule
ligand_mol = plf.Molecule(rdkit_ligand)

print("Calculating 3D interaction fingerprints...")

# 4. Run the fingerprint processing directly on the two separate molecule structures
fp = plf.Fingerprint()
fp.run_from_molecules(ligand_mol=[ligand_mol], receptor_mol=receptor_mol)

# 5. Generate the interactive 2D diagram map view
net = fp.plot_network(frame=0, display_residues="all")

# 6. Save the final 2D diagram view as an HTML file
output_html = "ligand_130275249_interaction_map.html"
net.save(output_html)

print(f"\n🎉 Success! Your 2D interaction map has been saved as: {os.path.abspath(output_html)}")
print("Double-click the .html file to open and view your interaction network!")

Parsing structures using ProLIF native readers...


d:\PriceTracker\ng\Lib\site-packages\MDAnalysis\converters\RDKit.py:500: UserWarning: No hydrogen atom found in the topology. Forcing to continue the conversion.
  warnings.warn(


: 

In [18]:
import os
import MDAnalysis as mda
import prolif as plf

# 1. Define paths to your receptor and your docked ligand
# (Make sure to replace receptor.pdb with your actual protein file)
receptor_path = r"D:\PriceTracker\ERBB2\data\receptor\3RCD.pdb" 
ligand_path = r"D:\PriceTracker\ERBB2\data\prepaerd_ligands_pdb\ligand_130275249_temp.sd"

if not os.path.exists(receptor_path) or not os.path.exists(ligand_path):
    raise FileNotFoundError("Please ensure both your receptor.pdb and ligand file paths are correct.")

print("Loading structures and calculating 3D interaction contacts...")

# 2. Load the protein and ligand into an MDAnalysis Universe
u = mda.Universe(receptor_path, ligand_path)

# 3. Create selections for ProLIF
# 'protein' selects the receptor; 'not protein' grabs your ligand molecule
receptor_selection = u.select_atoms("protein")
ligand_selection = u.select_atoms("not protein")

# 4. Calculate the non-bonded interactions (H-bonds, pi-stacking, hydrophobic)
fp = plf.Fingerprint()
fp.run(u.trajectory, ligand_selection, receptor_selection)

# 5. Generate the interactive 2D interaction diagram network
# This generates a flat schematic showing the ligand structure and residue clouds
net = fp.plot_network(frame=0, display_residues="all")

# 6. Save the final 2D diagram view as an HTML file
output_html = "ligand_130275249_interaction_map.html"
net.save(output_html)

print(f"🎉 Success! Your 2D interaction map has been saved as: {os.path.abspath(output_html)}")
print("Double-click the .html file to open and view the diagram in your web browser.")

Loading structures and calculating 3D interaction contacts...


TypeError: Cannot find an appropriate coordinate reader for file 'D:\PriceTracker\ERBB2\data\prepaerd_ligands_pdb\ligand_130275249_temp.sd'.
           Unknown coordinate trajectory format 'SD' for 'D:\PriceTracker\ERBB2\data\prepaerd_ligands_pdb\ligand_130275249_temp.sd'. The FORMATs 
           dict_keys(['CHAIN', 'CHEMFILES', 'CRD', 'DCD', 'CONFIG', 'HISTORY', 'DMS', 'GMS', 'GRO', 'IMD', 'INPCRD', 'RESTRT', 'LAMMPS', 'DATA', 'LAMMPSDUMP', 'MOL2', 'PDB', 'ENT', 'XPDB', 'PDBQT', 'PQR', 'TPR', 'TRC', 'TRJ', 'MDCRD', 'CRDBOX', 'NCDF', 'NC', 'TRR', 'H5MD', 'TRZ', 'XTC', 'XYZ', 'TXYZ', 'ARC', 'MEMORY', 'MMTF', 'GSD', 'COOR', 'NAMDBIN', 'IN', 'FHIAIMS', 'TNG', 'PARMED', 'RDKIT', 'OPENMMSIMULATION', 'OPENMMAPP'])
           are implemented in MDAnalysis.
           See https://docs.mdanalysis.org/documentation_pages/coordinates/init.html#id1
           Use the format keyword to explicitly set the format: 'Universe(...,format=FORMAT)'
           For missing formats, raise an issue at https:/

In [1]:
import os
import MDAnalysis as mda
import prolif as plf
from rdkit import Chem

# 1. Define paths to your receptor and your docked ligand
receptor_path = r"D:\PriceTracker\ERBB2\data\receptor\3RCD.pdb" 
ligand_path = r"D:\PriceTracker\ERBB2\data\prepaerd_ligands_pdb\ligand_130275249_temp.sd"

if not os.path.exists(receptor_path) or not os.path.exists(ligand_path):
    raise FileNotFoundError("Please ensure both your receptor.pdb and ligand file paths are correct.")

print("Parsing structures using ProLIF native readers...")

# 2. Load the receptor into a ProLIF Molecule object
# FIXED: Added force=True to bypass the explicit hydrogen structural check
u_receptor = mda.Universe(receptor_path)
receptor_mol = plf.Molecule.from_mda(u_receptor, force=True)

# 3. Load the ligand from the .sd file using RDKit, then wrap with ProLIF
supplier = Chem.SDMolSupplier(ligand_path)
rdkit_ligand = supplier[0] if supplier else None
if rdkit_ligand is None:
    raise ValueError("RDKit failed to read the .sd file. Ensure it is a valid SDF format.")

# Convert RDKit object safely into a ProLIF molecule
ligand_mol = plf.Molecule(rdkit_ligand)

print("Calculating 3D interaction fingerprints...")

# 4. Run the fingerprint processing directly on the two separate molecule structures
fp = plf.Fingerprint()
fp.run_from_molecules(ligand_mol=[ligand_mol], receptor_mol=receptor_mol)

# 5. Generate the interactive 2D diagram map view
net = fp.plot_network(frame=0, display_residues="all")

# 6. Save the final 2D diagram view as an HTML file
output_html = "ligand_130275249_interaction_map.html"
net.save(output_html)

print(f"\n🎉 Success! Your 2D interaction map has been saved as: {os.path.abspath(output_html)}")
print("Double-click the .html file to open and view your interaction network!")

d:\PriceTracker\ng\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
d:\PriceTracker\ng\Lib\site-packages\MDAnalysis\topology\tables.py:52: DeprecationWarning: Deprecated in version 2.8.0
MDAnalysis.topology.tables has been moved to MDAnalysis.guesser.tables. This import point will be removed in MDAnalysis version 3.0.0
  warnings.warn(wmsg, category=DeprecationWarning)


Parsing structures using ProLIF native readers...


d:\PriceTracker\ng\Lib\site-packages\MDAnalysis\converters\RDKit.py:500: UserWarning: No hydrogen atom found in the topology. Forcing to continue the conversion.
  warnings.warn(


: 